# Method A: learn the experts' scores

**Paper output being replaced:** Table 10. Each industry gets a score (low = more polluting) and is ranked within its industrial area.
`score = Σ criterion_weight(C1..C10) × local_weight(industry, criterion)`. The experts produced the local weights (Table 6 ratings, then fuzzy AHP).

Per-industry summary (`wqml.summarise`): one row per industry, one column per paper criterion, all computed from lab data.

| Criterion | Summary value | Worse when |
|---|---|---|
| C1 Effluent discharge | mean flow (MLD) | higher |
| C2 ETP/CETP efficiency | mean (inlet − outlet) / inlet | lower |
| C3 / C4 Critical parameters, pre- / post-monsoon | pollutants over the CPCB limit per sampling event | higher |
| C5 Effluent type | industry type → paper's 1–9 scale (Table 4) | lower |
| C6 / C7 River impact, pre- / post-monsoon | downstream − upstream, percentile within pollutant | higher |
| C8 / C9 Groundwater impact, pre- / post-monsoon | near well − control well, percentile within pollutant | higher |
| C10 Public health | Σ toxicity × amount over the limit, per event | higher |

**Method A:** a regression model from the summary to the paper's score (divided by its area's mean). The experts are needed once, for the 33 training labels. After that, any new industry is scored from measurements alone. It learns what the experts would have said, so it can't be better than them.

**Which model?** Twelve scikit-learn regressors are compared on the same data, each tested leave-one-out. Picking the best of 12 on 33 rows flatters the winner, so nested cross-validation estimates how well the whole *pick-the-best* procedure really does.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV, HuberRegressor, LassoCV, LinearRegression, RidgeCV
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVR
import wqml
from wqml import CRITERIA
warnings.filterwarnings("ignore", category=UserWarning)

## 1. Data and per-industry summary

In [ ]:
raw, SOURCE = wqml.load()  # your data, else the paper's source report (data/uppcb/), else dummy
DUMMY = SOURCE == "dummy"
print({"user": "Your data.", "uppcb": "Paper's source data: UPPCB/NIH 2013 report, 54 stations, May and Nov 2012.",
       "dummy": "DUMMY data: results show the code works, nothing about the real industries."}[SOURCE])
df = wqml.prepare(raw)
summary = wqml.summarise(df)
labels = wqml.paper_labels()  # Table 10 score and target = score / area mean
summary.round(3)

In [ ]:
train = summary.join(labels, how="inner")
X, fuzzy, area = train[CRITERIA], train["fuzzy_weight"], train["industrial_area"]
y_all = train["target"]  # score / area mean over ALL labels: only for the final fit and plots, never for testing
print(len(train), "labelled industries")

## 2. Candidate models
Same preprocessing for all: fill gaps with the median, compress heavy tails (arcsinh), standardise. Settings are deliberately conservative for 33 rows; the linear models tune their penalty internally.

In [ ]:
CANDIDATES = {
    "always average (baseline)": DummyRegressor(),
    "linear regression": LinearRegression(),
    "ridge": RidgeCV(alphas=np.logspace(-2, 3, 30)),
    "lasso": LassoCV(cv=5, random_state=0),
    "elastic net": ElasticNetCV(cv=5, l1_ratio=[.2, .5, .8], random_state=0),
    "huber (robust linear)": HuberRegressor(alpha=1, max_iter=1000),
    "PLS (2 components)": PLSRegression(2),
    "k-nearest neighbours": KNeighborsRegressor(5, weights="distance"),
    "SVR (rbf)": SVR(C=1, epsilon=0.1),
    "random forest": RandomForestRegressor(300, min_samples_leaf=3, random_state=0, n_jobs=-1),
    "extra trees": ExtraTreesRegressor(300, min_samples_leaf=3, random_state=0, n_jobs=-1),
    "gradient boosting": GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05,
                                                   subsample=0.8, random_state=0),
    "gaussian process": GaussianProcessRegressor(ConstantKernel() * RBF(np.ones(len(CRITERIA))) + WhiteKernel(),
                                                 normalize_y=True, random_state=0),
}


def pipe(est):
    return make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                         FunctionTransformer(np.arcsinh), StandardScaler(), est)


def cv_predict(est, X, fuzzy, area, cv):
    """Out-of-fold predictions. The target is normalised with TRAINING labels only:
    y = score / mean score of the training industries in the same area. The held-out industry's
    own score never enters its training target. Returns (prediction, truth on the same scale)."""
    pred = pd.Series(np.nan, index=X.index)
    truth = pd.Series(np.nan, index=X.index)
    for tr, te in cv.split(X):
        m = fuzzy.iloc[tr].groupby(area.iloc[tr]).mean()
        fit = pipe(clone(est)).fit(X.iloc[tr], fuzzy.iloc[tr] / area.iloc[tr].map(m))
        pred.iloc[te] = np.ravel(fit.predict(X.iloc[te]))
        truth.iloc[te] = fuzzy.iloc[te] / area.iloc[te].map(m)
    return pred, truth


def evaluate(pred, truth):
    """Error on the fold-normalised target, plus rank agreement with the paper within each area."""
    ag = wqml.agreement(train.assign(p=pred), "p")
    return {"MAE": abs(pred - truth).mean(), "spearman_within_area": ag["spearman_within_area"],
            "same_worst_industry": ag["same_worst_industry"]}

## 3. Compare (leave-one-out)
Each industry is predicted by a model trained on the other 32, with the target normalised from those 32 only. Lower MAE is better; Spearman = rank agreement with the paper within each area (1 = same order). The baseline's leave-one-out predictions are anti-correlated with the truth by construction, so its Spearman is blanked.

In [ ]:
loo = {name: cv_predict(est, X, fuzzy, area, LeaveOneOut()) for name, est in CANDIDATES.items()}
results = pd.DataFrame({name: evaluate(*pt) for name, pt in loo.items()}).T.sort_values("MAE")
results.loc["always average (baseline)", "spearman_within_area"] = np.nan
results.round(3)

## 4. Honest estimate: nested cross-validation
For each held-out industry, pick the model with the lowest MAE using only the other 32 (inner 5-fold), then predict the held-out one. The score describes the *procedure* "take the best of these 12", free of selection optimism. It also shows how often each model wins, i.e. how stable the choice is. Takes a few minutes.

In [ ]:
def pick_best(idx):
    inner = KFold(5, shuffle=True, random_state=0)
    mae = {}
    for n, e in CANDIDATES.items():
        if n == "always average (baseline)":
            continue
        p, tr = cv_predict(e, X.loc[idx], fuzzy.loc[idx], area.loc[idx], inner)
        mae[n] = abs(p - tr).mean()
    return min(mae, key=mae.get)

nested_pred, nested_truth, picked = pd.Series(index=X.index, dtype=float), pd.Series(index=X.index, dtype=float), []
for tr, te in LeaveOneOut().split(X):
    name = pick_best(X.index[tr])
    picked.append(name)
    m = fuzzy.iloc[tr].groupby(area.iloc[tr]).mean()
    fit = pipe(clone(CANDIDATES[name])).fit(X.iloc[tr], fuzzy.iloc[tr] / area.iloc[tr].map(m))
    nested_pred.iloc[te] = np.ravel(fit.predict(X.iloc[te]))
    nested_truth.iloc[te] = fuzzy.iloc[te] / area.iloc[te].map(m)
print("Nested estimate:", {k: round(v, 3) for k, v in evaluate(nested_pred, nested_truth).items()})
pd.Series(picked).value_counts().rename("times picked")

## 5. Chosen model
Rule: the lowest leave-one-out MAE. When the top models are within noise of each other (see the table), they behave alike and any of them is a sound choice; prefer the simplest.

In [ ]:
CHOSEN = results.index[0]
model = pipe(clone(CANDIDATES[CHOSEN]))
train["A_score"] = loo[CHOSEN][0]
print("Chosen:", CHOSEN)
wqml.agreement(train, "A_score")

## 6. What the model relies on
Permutation importance: how much the error grows when one criterion is shuffled. Measured on the training data, so it describes the fitted model, not a causal effect.

In [ ]:
model.fit(X, y_all)
imp = permutation_importance(model, X, y_all, scoring="neg_mean_absolute_error", n_repeats=50, random_state=0)
pd.Series(imp.importances_mean, index=CRITERIA).round(4).sort_values(ascending=False)

## 7. Ranking: paper vs A

In [ ]:
train["fuzzy_rank"] = train.groupby("industrial_area")["fuzzy_weight"].rank(method="min").astype(int)
train["A_rank"] = train.groupby("industrial_area")["A_score"].rank(method="min").astype(int)
train[["industrial_area", "fuzzy_rank", "A_rank", "target", "A_score"]].sort_values(["industrial_area", "fuzzy_rank"]).round(3)

In [ ]:
plt.scatter(train["A_score"], train["target"])
plt.axline((1, 1), slope=1, ls="--", c="grey")
plt.xlabel(f"A prediction ({CHOSEN}, leave-one-out)"); plt.ylabel("paper score / area mean")
plt.title("Method A" + (" (dummy)" if DUMMY else ""));

## 8. Score a new industry
The chosen model, fitted on all 33 industries, predicts any summary row, e.g. a new industry's lab data run through `wqml.summarise`.

In [ ]:
new = summary.head(3)  # replace with a new industry's summary rows
pd.Series(np.ravel(model.predict(new[CRITERIA])), index=new.index,
          name="A_score (1 = area average, lower = worse)").round(3)

## Before relying on it
- [ ] Confirm the INFERRED Magarwara station mapping in `data/uppcb/stations.csv`
- [ ] Nested estimate clearly better than always-average
- [ ] Compare with method B: if B matches the experts as well or better with no training, prefer B
- [ ] More expert-scored industries if possible: each new label is worth more than more samples